In [ ]:
-- Create Customers table
CREATE OR REPLACE TABLE CUSTOMERS (
    CUSTOMER_ID INT,
    NAME STRING,
    AGE INT,
    COUNTRY STRING,
    JOIN_DATE DATE
);

INSERT INTO CUSTOMERS VALUES
(1, 'MANDAR', 45, 'USA', '2023-01-15'),
(2, 'PRADEEP', 45, 'UK', '2023-03-10'),
(3, 'VINOD', 40, 'USA', '2022-11-05'),
(4, 'MATT', 45, 'CANADA', '2023-06-20'),
(5, 'ASHISH', 40, 'USA', '2023-08-01');

-- Create Transactions table
CREATE OR REPLACE TABLE TRANSACTIONS (
    TRANSACTION_ID INT,
    CUSTOMER_ID INT,
    AMOUNT FLOAT,
    TRANSACTION_DATE DATE,
    FRAUD_FLAG INT
);

INSERT INTO TRANSACTIONS VALUES
(101, 1, 500.00, '2023-05-01', 0),
(102, 1, 12000.00, '2023-06-15', 1),
(103, 2, 200.00, '2023-07-01', 0),
(104, 3, 15000.00, '2023-07-20', 1),
(105, 4, 300.00, '2023-08-05', 0),
(106, 5, 100.00, '2023-08-10', 0),
(107, 3, 1000.00, '2023-07-20', 1),
(108, 4, 3200.00, '2023-08-05', 0),
(109, 5, 1040.00, '2023-08-10', 0),
(110, 3, 150.00, '2023-07-20', 1),
(111, 4, 360.00, '2023-08-05', 0),
(112, 5, 1006.00, '2023-08-10', 0);


CREATE OR REPLACE TEMP TABLE CUSTOMER_FEATURES AS
SELECT 
    c.CUSTOMER_ID,
    AGE,
    COUNTRY,
    COUNT(t.TRANSACTION_ID) AS txn_count,
    SUM(t.AMOUNT) AS total_spent,
    MAX(t.TRANSACTION_DATE) AS last_txn_date,
    MAX(t.FRAUD_FLAG) AS fraud_flag
FROM CUSTOMERS c
LEFT JOIN TRANSACTIONS t 
    ON c.CUSTOMER_ID = t.CUSTOMER_ID
GROUP BY c.CUSTOMER_ID, AGE, COUNTRY;

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.modeling.linear_model import LogisticRegression
from snowflake.ml.modeling.preprocessing import OneHotEncoder

session = get_active_session()

df = session.table("CUSTOMER_FEATURES")

encoder = OneHotEncoder(input_cols=["COUNTRY"], output_cols=["COUNTRY_ENC"])
encoded_df = encoder.fit(df).transform(df)
encoded_cols = [c for c in encoded_df.columns if c.startswith("COUNTRY_ENC_")]

lr = LogisticRegression(
    input_cols=["AGE", "TXN_COUNT", "TOTAL_SPENT"] + encoded_cols,
    label_cols=["FRAUD_FLAG"]
)

model = lr.fit(encoded_df)

predictions = model.predict(encoded_df)
predictions.write.mode("overwrite").save_as_table("PREDICTIONS_TABLE")
predictions.show()

In [ ]:
CREATE OR REPLACE TABLE CUSTOMER_RISK AS
SELECT 
    CUSTOMER_ID,
    "OUTPUT_FRAUD_FLAG" AS risk_level
FROM PREDICTIONS_TABLE;

SELECT * FROM CUSTOMER_RISK;